# Modelling
In this notebook, we will test out various models on the dataset with SMOTENC and k-Nearest Neighbors imputation

Before modelling, the numerical data will be scaled using the MinMaxScaler because these features operate on different units, and have important outliers.

In [1]:
# Import libraries
import pandas as pd
import numpy as np

# Import dataset
df = pd.read_csv('../data/smotenc_knn_df.csv') # k-Nearest Neighbors imputed dataset with SMOTENC

In [2]:
# Import train, test, split
from sklearn.model_selection import train_test_split

# List of continuous features
cont_features = ['age', 'cigsPerDay', 'totChol', 'BMI', 'heartRate', 'glucose' ,'MAP']

# Define features and target
X = df.drop(columns='TenYearCHD')
y = df['TenYearCHD']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
# Import libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

# Define models
models = {
    'KNN': KNeighborsClassifier(),
    'LogisticRegression' : LogisticRegression(max_iter=1000),
    'RandomForest' : RandomForestClassifier(random_state=42)
}

In order to determine which model performs the best, we will define a function that allows us to assess model performance using AUC-ROC as a metric, and stratified k-fold as a technique.

In [7]:
# Import Libraries
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Define a function to evaluate model performance
def model_eval(model, X, y):
    # Apply k-fold cross validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')

    # Print performance metrics
    print(f'Mean ROC-AUC: {scores.mean():.2f}')
    print(f'Standard Deviation: {scores.std():.2f}')

In [8]:
# Import libraries
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

# Scale the continuous data
scaler = MinMaxScaler()
X_train[cont_features] = scaler.fit_transform(X_train[cont_features])
X_test[cont_features] = scaler.fit_transform(X_test[cont_features])

In [9]:
# Evaluate the models
for name, model in models.items():
    print(f'--Evaluating {name}--')
    pipeline = Pipeline([
        ('scaler', MinMaxScaler()),
        ('model', model)
    ])

    # Evaluate model performance
    model_eval(pipeline, X_train, y_train)

    # Fit and predict on a test set
    pipeline.fit(X_train, y_train,)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    # Report and confusion matrix
    print(classification_report(y_test, y_pred))
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, y_pred))

    # ROC-AUC score
    roc_auc = roc_auc_score(y_test, y_proba)
    print(f'ROC-AUC on test set: {roc_auc:.2f}\n')

--Evaluating KNN--
Mean ROC-AUC: 0.84
Standard Deviation: 0.01
              precision    recall  f1-score   support

           0       0.75      0.71      0.73       561
           1       0.72      0.76      0.74       557

    accuracy                           0.74      1118
   macro avg       0.74      0.74      0.74      1118
weighted avg       0.74      0.74      0.74      1118

Confusion Matrix:
[[399 162]
 [131 426]]
ROC-AUC on test set: 0.80

--Evaluating LogisticRegression--
Mean ROC-AUC: 0.75
Standard Deviation: 0.02
              precision    recall  f1-score   support

           0       0.70      0.66      0.68       561
           1       0.68      0.72      0.70       557

    accuracy                           0.69      1118
   macro avg       0.69      0.69      0.69      1118
weighted avg       0.69      0.69      0.69      1118

Confusion Matrix:
[[368 193]
 [154 403]]
ROC-AUC on test set: 0.74

--Evaluating RandomForest--
Mean ROC-AUC: 0.96
Standard Deviation: 0.